# EvoGraph-R1 (rút gọn) — chạy lại trên Colab
Chạy lần lượt từng ô. **Runtime → Change runtime type → T4 GPU** (nếu dùng mô hình chạy cục bộ).
Hai cách chọn LLM, không cần Groq:
- `local` (khuyến nghị): chạy Qwen2.5-7B-Instruct-AWQ bằng vLLM ngay trên GPU T4 miễn phí, không cần API key.
- `gemini`: dùng Gemini API miễn phí (khóa tại https://aistudio.google.com/apikey), lưu vào Colab Secrets với tên `GEMINI_API_KEY`. Hạn mức miễn phí thấp nên chỉ hợp để chạy thử.

In [ ]:
BACKEND = "local"   # "local" hoặc "gemini"
N = 100             # số câu hỏi; đặt 5 để chạy thử

## 1. Cài đặt và lấy mã

In [ ]:
!git clone -q https://github.com/FluteInTheRain/evograph-reproduction.git
%cd evograph-reproduction
!pip install -q -r requirements.txt gdown

## 2. Đăng nhập W&B

In [ ]:
import wandb
wandb.login()

## 3. Dữ liệu MetaQA
Thử tải tự động; nếu Google Drive từ chối, hãy tải tay `kb.txt` và `2-hop/vanilla/qa_test.txt` từ https://github.com/yuyuz/MetaQA rồi upload vào thư mục `data/` (đặt tên `kb.txt`, `2hop_qa_test.txt`).

In [ ]:
import os
os.makedirs("data", exist_ok=True)
if not (os.path.exists("data/kb.txt") and os.path.exists("data/2hop_qa_test.txt")):
    !gdown --folder https://drive.google.com/drive/folders/0B-36Uca2AvwhTWVFSUZqRXVtbUE -O data/metaqa
    !cp data/metaqa/kb.txt data/kb.txt
    !cp data/metaqa/2-hop/vanilla/qa_test.txt data/2hop_qa_test.txt
!wc -l data/kb.txt data/2hop_qa_test.txt

## 4. Chọn LLM

In [ ]:
import subprocess, time, os, requests
if BACKEND == "local":
    !pip install -q vllm
    server = subprocess.Popen(
        ["python", "-m", "vllm.entrypoints.openai.api_server",
         "--model", "Qwen/Qwen2.5-7B-Instruct-AWQ", "--quantization", "awq",
         "--max-model-len", "6144", "--gpu-memory-utilization", "0.9", "--port", "8000"],
        stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(120):
        try:
            requests.get("http://localhost:8000/v1/models", timeout=2); break
        except Exception:
            time.sleep(5)
    os.environ["LLM_KEY"] = "EMPTY"
    LLM = "--model Qwen/Qwen2.5-7B-Instruct-AWQ --base_url http://localhost:8000/v1 --api_key_env LLM_KEY"
else:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
    LLM = "--model gemini-2.5-flash-lite --base_url https://generativelanguage.googleapis.com/v1beta/openai/ --api_key_env GEMINI_API_KEY"
print(LLM)

## 5. Chạy thử nhanh (5 câu)

In [ ]:
!python run.py --kb data/kb.txt --qa data/2hop_qa_test.txt --variant full --n 5 --wandb_group smoke-test $LLM

## 6. Bốn cấu hình chính

In [ ]:
for v in ["static", "web", "edit", "full"]:
    !python run.py --kb data/kb.txt --qa data/2hop_qa_test.txt --variant $v --n $N --wandb_group main-ablation $LLM

## 7. Ablation theo mức khuyết của KG

In [ ]:
for d in [0.0, 0.25, 0.5, 0.75]:
    for v in ["static", "full"]:
        !python run.py --kb data/kb.txt --qa data/2hop_qa_test.txt --variant $v --n $N --drop $d --wandb_group ablation-drop $LLM

## 8. Ablation theo độ nhiễu của web

In [ ]:
for p in [0.0, 0.1, 0.3, 0.5]:
    !python run.py --kb data/kb.txt --qa data/2hop_qa_test.txt --variant full --n $N --web_noise $p --wandb_group ablation-webnoise $LLM